In [1]:
!pip install --user -U pip huggingface_hub accelerate bitsandbytes
!pip install --user "transformers==4.57.6"

  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-2.4.0-py3-none-any.whl (857 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
tokenizers 0.22.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.4.0 which is incompatible.
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninsta

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

CUDA available: True
GPU count: 3
  cuda:0 -> NVIDIA A100-PCIE-40GB
  cuda:1 -> NVIDIA A100-PCIE-40GB
  cuda:2 -> NVIDIA A100-PCIE-40GB


In [3]:
import psutil
import shutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# RAM
ram = psutil.virtual_memory()

print("=== SYSTEM RAM ===")
print(f"Total RAM     : {ram.total / (1024**3):.2f} GB")
print(f"Available RAM : {ram.available / (1024**3):.2f} GB")
print(f"Used RAM      : {ram.used / (1024**3):.2f} GB")

# Disk
total, used, free = shutil.disk_usage("/")

print("\n=== SYSTEM DISK ===")
print(f"Total Disk : {total / (1024**3):.2f} GB")
print(f"Used Disk  : {used / (1024**3):.2f} GB")
print(f"Free Disk  : {free / (1024**3):.2f} GB")


=== SYSTEM RAM ===
Total RAM     : 499.07 GB
Available RAM : 469.77 GB
Used RAM      : 29.30 GB

=== SYSTEM DISK ===
Total Disk : 435.54 GB
Used Disk  : 220.00 GB
Free Disk  : 193.34 GB


In [4]:
!nvidia-smi
!df -h ~
!free -h

Fri Nov 13 20:46:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:21:00.0 Off |                    0 |
| N/A   29C    P0             45W /  250W |     427MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    print(f"cuda:{i}: {allocated:.2f} GB allocated")

cuda:0: 0.00 GB allocated
cuda:1: 0.00 GB allocated
cuda:2: 0.00 GB allocated


In [5]:
PROMPT = "2+2="
GEN_KWARGS = dict(max_new_tokens=32, do_sample=False)

In [6]:
print("loading Qwen2.5-Coder-7B...")
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
qwen_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    # quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)
inputs = qwen_tok(PROMPT, return_tensors="pt").to(qwen_model.device)
output = qwen_model.generate(**inputs, **GEN_KWARGS)
print("DONE:", qwen_tok.decode(output[0], skip_special_tokens=True))

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    print(f"cuda:{i}: {allocated:.2f} GB allocated")

loading Qwen2.5-Coder-7B...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


DONE: 2+2=!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
cuda:0: 8.12 GB allocated
cuda:1: 10.43 GB allocated
cuda:2: 9.85 GB allocated


In [7]:
print("loading DeepSeek-Coder-V2-Lite-Instruct...")
deepseek_tok = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct")
deepseek_model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct",
    # quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map= "auto"
)
inputs = deepseek_tok(PROMPT, return_tensors="pt").to(deepseek_model.device)
output = deepseek_model.generate(**inputs, **GEN_KWARGS)
print("DONE:", deepseek_tok.decode(output[0], skip_special_tokens=True))

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    print(f"cuda:{i}: {allocated:.2f} GB allocated")

loading DeepSeek-Coder-V2-Lite-Instruct...


`rope_scaling`'s factor field must be a float >= 1, got 40
`rope_scaling`'s beta_fast field must be a float, got 32
`rope_scaling`'s beta_slow field must be a float, got 1


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


DONE: 2+2=!!!!!!!!!!!!!!!!!!!!!!(!!!!!!!!!
cuda:0: 24.45 GB allocated
cuda:1: 30.04 GB allocated
cuda:2: 32.42 GB allocated


In [8]:
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
for name in os.listdir(cache_dir):
    if name.startswith("models--"):
        print(name)

models--deepseek-ai--DeepSeek-Coder-V2-Lite-Instruct
models--Qwen--Qwen2.5-Coder-7B-Instruct
